# **Stokes and Supremizer Stabilization**
In a Stokes problem, in order to guarantee the well-posedness, we have to prove:

1. The coercivity of the grad-grad form over the kernel of the divergence operator.
2. The divergence operator has to verify the inf-sup stability condition, namely:
$$
\beta_{\delta}(\mu) = \inf_{\mathsf q \neq 0} \sup_{\mathsf v \neq 0} 
\frac{\mathsf q^T B \mathsf v}{\vert \vert \mathsf q \vert \vert \vert \vert \mathsf v \vert \vert } > 0.
$$ 

The inf-sup relation means that $\mathsf{ker}(B^T) = \{0\}$ (no pressures different from zero let the divergence to be null).

In the FE setting, the above relation is verified with some special combinations of spaces, like the **Taylor-Hood** approximation: namely $\mathbb P^2$ elements for velocity and $\mathbb P^1$ for pressure (also called $\mathbb P^2- \mathbb P^1$ elements).

How does it translates in the reduced framework?

Property 1. is directly inherited from the FE discretization, and property 2. reads:

$$
\beta_{N}(\mu) = \inf_{\mathsf q_{N_p} \neq 0} \sup_{\mathsf v_{N_u} \neq 0} 
\frac{\mathsf q_{N_p}^T B_{N} \mathsf v_{N_u}}{\vert \vert \mathsf q \vert \vert \vert \vert \mathsf v \vert \vert } > 0.
$$ 

It is clear that, in general, for reduced problems, the inf-sup condition is not fulfilled. Let us consider a reduced basis $\{\phi_i\}_{i=1}^{N_u}$ and $\{\psi_i\}_{i=1}^{N_p}$, for velocity and pressure, respectively. 
The matrix $[B_N]_{kj} = b(\phi_k, \psi_j) = 0$, since the velocity basis functions are conbination of weakly-divergence-free snapshots.

Namely $\mathsf{ker}(B_N^T) \neq \{0\}$, indeed, it is definetely larger, since every pressure basis is actually an element of the kernel!

To avoid this problem we use _supremizer stabilization_. 

From now on, we suppose $N_p = N_u = N$, just for a simpler presentation of the topic.

The trick is to enrich the velocity space with the following vector field $\mathsf s^{\mu}$ for each pressure mode and each related parametric instance:
$$
\mathbb X_u \mathsf s^{\mu}(\overline{\psi}_j) = B^T\overline{\psi}_j,
$$
where $\overline{\psi}_j$ is the FE vector related to the pressure basis. The vector field $\mathsf s^{\mu}(q)$ is called **supremizer** and it is defined as

$$
\mathsf  s^{\mu}(q) = {\arg \sup}_{\mathsf v \neq 0} 
\frac{\mathsf q_N^T B \mathsf v}{\vert \vert \mathsf v \vert \vert }.
$$

Indeed, in this way, we can prove the following relation:

$$
0 < \beta_{\delta}(\mu)  = \inf_{\mathsf q \neq 0} \sup_{\mathsf v \neq 0} 
\frac{\mathsf q^T B \mathsf v}{\vert \vert \mathsf q \vert \vert \vert \vert \mathsf v \vert \vert } \leq 
\inf_{\mathsf q_N \neq 0} \sup_{\mathsf v \neq 0} 
\frac{(\mathbb B_p\mathsf q_N)^T B \mathsf v}{\vert \vert \mathbb B_p\mathsf q_N \vert \vert \vert \vert \mathsf v \vert \vert } \leq 
\inf_{\mathsf q_N \neq 0} 
 \frac{(\mathbb B_p\mathsf q_N)^T B \mathsf s^{\mu}(\mathbb B_p\mathsf q_N)}{\vert \vert \mathbb B_p\mathsf q_N \vert \vert \vert \vert \mathsf s^{\mu}(\mathbb B_p\mathsf q_N)\vert \vert } \leq
 \inf_{\mathsf q_N \neq 0} \sup_{\mathsf v_N \neq 0} 
\frac{\mathsf q_N^T\mathbb B_p^T B  \mathbb B_u \mathsf v_N}{\vert \vert \mathbb B_p\mathsf q_N \vert \vert \vert \vert \mathbb B_u \mathsf v_N \vert \vert } = \inf_{\mathsf q_N \neq 0} \sup_{\mathsf v_N \neq 0} 
\frac{\mathsf q_N^T B_N\mathsf v_N}{\vert \vert \mathsf q_N \vert \vert \vert \vert \mathsf v_N \vert \vert} = \beta_{N}(\mu).
$$ 

Let us understand how to exploit these notions on the creation of the reduced space.

**Be careful**: you can see that the supremizer stabilization depends _online_ on the FOM dimension $N_{\delta}$.

In [ ]:
import numpy as np
import argparse
import os.path
import scipy.sparse
import vtk
from pypolydim import polydim, gedim
from pypolydim.export_vtk_utilities import ExportVTKUtilities
from pypolydim.assembler_utilities import assembler_utilities
import matplotlib

import sys
sys.path.insert(1, '../')
import other_utilities as other_ut

In [ ]:
geometry_utilities_config = gedim.GeometryUtilitiesConfig()
geometry_utilities_config.tolerance1_d = 1.0e-6
geometry_utilities_config.tolerance2_d = 1.0e-12
geometry_utilities = gedim.GeometryUtilities(geometry_utilities_config)
mesh_utilities = gedim.MeshUtilities()
vtk_utilities = ExportVTKUtilities()

## The parametric version of the Stokes problem

Solve the Stokes equation on square ${\Omega} = (0, 1) \times (0, 1)$

$$
\begin{cases}
-\mu_1 \nabla \cdot (\nabla \mathbf{u}) + \nabla p = \mathbf{f} & \text{in } \Omega\\
(\nabla \cdot \mathbf{u}) = 0 & \text{in } \Omega\\
u = 0 & \text{in } ∂ \Omega
\end{cases}
$$

where $\nu$ is the **viscosity**, $\mathbf{u} = (u_1, u_2)$ is the **speed** and $p$ is the **pressure**, with $\mathbf{f}$ a parametric version of the forcing term of Lab10.

In [ ]:
# Export folder
export_file_path = "./Export/Test_1"
if not os.path.exists(export_file_path):
    os.makedirs(export_file_path)

# Mesh file path
export_mesh_path = export_file_path + "/Mesh"
if not os.path.exists(export_mesh_path):
    os.makedirs(export_mesh_path)

# Solution file path
export_solution_path = export_file_path + "/Solution"
if not os.path.exists(export_solution_path):
    os.makedirs(export_solution_path)

In [ ]:
def Stokes_V():
	return 1.0

def Stokes_v(numPoints, points):
	values = np.ones(numPoints) * Stokes_V()
	return values.ctypes.data

def Stokes_advection_1(numPoints, points):
	values = np.zeros((2, numPoints), order='F')
	values[0,:] = 1.0
	return values.ctypes.data

def Stokes_advection_2(numPoints, points):
	values = np.zeros((2, numPoints), order='F')
	values[1,:] = 1.0
	return values.ctypes.data
############## Forcing term WRT mu_2 #####################
def Stokes_f_1(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = - ((mu_2**3) * np.pi * np.pi * np.cos((mu_2**2)* np.pi * matPoints[0,:]) - (mu_2**2) * np.pi * np.pi) * np.sin(mu_2 * np.pi * matPoints[1,:]) * np.cos(mu_2 * np.pi * matPoints[1,:]) + (+mu_2 * np.pi * np.cos(mu_2 * np.pi * matPoints[0,:]) * np.cos(mu_2 * np.pi * matPoints[1,:]))
	return values.ctypes.data

def Stokes_f_2(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = - (-(mu_2**3) * np.pi * np.pi * np.cos((mu_2**2) * np.pi * matPoints[1,:]) + (mu_2**2) * np.pi * np.pi) * np.sin(mu_2 * np.pi * matPoints[0,:]) * np.cos(mu_2 * np.pi * matPoints[0,:]) + (-mu_2* np.pi * np.sin(mu_2 * np.pi * matPoints[0,:]) * np.sin(mu_2 * np.pi * matPoints[1,:]))
	return values.ctypes.data
#############
def Stokes_pressure_exactSolution(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = np.sin(2.0 * np.pi * matPoints[0,:]) * np.cos(2.0 * np.pi * matPoints[1,:])
	return values.ctypes.data

def Stokes_speed_exactSolution_1(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = +0.5 * np.sin(2.0 * np.pi * matPoints[0,:]) * np.sin(2.0 * np.pi * matPoints[0,:]) * np.sin(2.0 * np.pi * matPoints[1,:]) * np.cos(2.0 * np.pi * matPoints[1,:])
	return values.ctypes.data

def Stokes_speed_exactSolution_2(numPoints, points):
	matPoints = gedim.make_nd_matrix(points, (3, numPoints), np.double)
	values = -0.5 * np.sin(2.0 * np.pi * matPoints[1,:]) * np.sin(2.0 * np.pi * matPoints[1,:]) * np.sin(2.0 * np.pi * matPoints[0,:]) * np.cos(2.0 * np.pi * matPoints[0,:])
	return values.ctypes.data

### Discretization

In [ ]:
pde_domain = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D()
pde_domain.vertices = np.array([[0.0, 1.0, 1.0, 0.0],
                                [0.0, 0.0, 1.0, 1.0],
                                [0.0, 0.0, 0.0, 0.0]])
pde_domain.shape_type = polydim.pde_tools.mesh.pde_mesh_utilities.PDE_Domain_2D.Domain_Shape_Types.parallelogram
pde_domain.area = 1.0

In [ ]:
mesh_type = polydim.pde_tools.mesh.pde_mesh_utilities.MeshGenerator_Types_2D.triangular
method_type = polydim.pde_tools.local_space_pcc_2_d.MethodTypes.fem_pcc
mesh_size = 0.001

In [ ]:
mesh_data = gedim.MeshMatrices()
mesh = gedim.MeshMatricesDAO(mesh_data)

polydim.pde_tools.mesh.pde_mesh_utilities.create_mesh_2_d(geometry_utilities,
                                                          mesh_utilities,
                                                          mesh_type,
                                                          pde_domain,
                                                          mesh_size,
                                                          mesh)
mesh_geometric_data = polydim.pde_tools.mesh.pde_mesh_utilities.compute_mesh_2_d_geometry_data(geometry_utilities, mesh_utilities, mesh)

In [ ]:
vtk_utilities.export_mesh(export_mesh_path, mesh)
other_ut.plot_mesh(mesh)

### High Fidelity approximation

In [ ]:
info_internal = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_internal.marker = 0

info_dirichlet = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.strong)
info_dirichlet.marker = 1

info_neumann_none = polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo(polydim.pde_tools.do_fs.DOFsManager.MeshDOFsInfo.BoundaryInfo.BoundaryTypes.none)
info_neumann_none.marker = 0

pressure_boundary_info = {
    0: info_internal,
    1: info_dirichlet,
    2: info_neumann_none,
    3: info_neumann_none,
    4: info_neumann_none,
    5: info_neumann_none,
    6: info_neumann_none,
    7: info_neumann_none,
    8: info_neumann_none
}

speed_boundary_info = {
    0: info_internal,
    1: info_dirichlet,
    2: info_dirichlet,
    3: info_dirichlet,
    4: info_dirichlet,
    5: info_dirichlet,
    6: info_dirichlet,
    7: info_dirichlet,
    8: info_dirichlet
}

In [ ]:
mesh_connectivity_data = polydim.pde_tools.mesh.MeshMatricesDAO_mesh_connectivity_data(mesh)

pressure_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, 
                                                                                              1)
speed_reference_element_data = polydim.pde_tools.local_space_pcc_2_d.create_reference_element(method_type, 
                                                                                             2)

dof_manager = polydim.pde_tools.do_fs.DOFsManager()

pressure_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(pressure_reference_element_data, 
                                                                                    mesh, 
                                                                                    pressure_boundary_info)
pressure_dofs_data = dof_manager.create_do_fs_2_d(pressure_mesh_dofs_info, 
                                                  mesh_connectivity_data)

speed_mesh_dofs_info = polydim.pde_tools.local_space_pcc_2_d.set_mesh_do_fs_info(speed_reference_element_data, 
                                                                                 mesh, 
                                                                                 speed_boundary_info)
speed_dofs_data = dof_manager.create_do_fs_2_d(speed_mesh_dofs_info, 
                                               mesh_connectivity_data)

In [ ]:
pressure_n_dofs = pressure_dofs_data.number_do_fs
pressure_n_strongs = pressure_dofs_data.number_strongs
speed_n_dofs = speed_dofs_data.number_do_fs
speed_n_strongs = speed_dofs_data.number_strongs
tot_dofs = 2 * speed_n_dofs + pressure_n_dofs
tot_strongs = 2 * speed_n_strongs + pressure_n_strongs

In [ ]:
print("P dofs\t", "P stgs\t", "U dofs\t", "U stgs\t", "T dofs\t", "T stgs")
print(pressure_n_dofs,"\t", pressure_n_strongs,"\t", speed_n_dofs,"\t", speed_n_strongs,"\t", tot_dofs,"\t", tot_strongs)

In [ ]:
def pressure_exact(x, y, z):
    return np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * y)

def pressure_exact_gradient(x, y, z):
    return np.array([\
        +2.0 * np.pi * np.cos(2.0 * np.pi * x) * np.cos(2.0 * np.pi * y),\
        -2.0 * np.pi * np.sin(2.0 * np.pi * x) * np.sin(2.0 * np.pi * y),\
        0.0])

def speed_x_exact(x, y, z):
    return +0.5 * np.sin(2.0 * np.pi * x) * np.sin(2.0 * np.pi * x) * np.sin(2.0 * np.pi * y) * np.cos(2.0 * np.pi * y)
def speed_y_exact(x, y, z):
    return -0.5 * np.sin(2.0 * np.pi * y) * np.sin(2.0 * np.pi * y) * np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * x)

def speed_x_exact_laplacian(x, y, z):
    return (+8.0 * np.pi * np.pi * np.cos(4.0 * np.pi * x) - 4.0 * np.pi * np.pi) * np.sin(2.0 * np.pi * y) * np.cos(2.0 * np.pi * y)
def speed_y_exact_laplacian(x, y, z):
    return (-8.0 * np.pi * np.pi * np.cos(4.0 * np.pi * y) + 4.0 * np.pi * np.pi) * np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * x)

In [ ]:
mu_2 = 2.

def f_x_function(x, y, z):
    return - ((mu_2**3) * np.pi * np.pi * np.cos((mu_2**2)* np.pi * x) - (mu_2**2) * np.pi * np.pi) * np.sin(mu_2 * np.pi * y) * np.cos(mu_2 * np.pi * y) + (+mu_2 * np.pi * np.cos(mu_2 * np.pi * x) * np.cos(mu_2 * np.pi * y))
def f_y_function(x, y, z):
    return - (-(mu_2**3) * np.pi * np.pi * np.cos((mu_2**2) * np.pi * y) + (mu_2**2) * np.pi * np.pi) * np.sin(mu_2 * np.pi * x) * np.cos(mu_2 * np.pi * x) + (-mu_2* np.pi * np.sin(mu_2 * np.pi * x) * np.sin(mu_2 * np.pi * y))

f_x = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_reference_element_data,
                                                                       f_x_function)
f_y = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_reference_element_data,
                                                                       f_y_function)

J_f = np.concatenate([f_x, f_y, np.zeros(pressure_n_dofs)])

In [ ]:
def pressure_strong_function(marker, x, y, z):  
    return pressure_exact(x, y, z)

p_strong = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       pressure_mesh_dofs_info,
                                                                       pressure_dofs_data,
                                                                       pressure_reference_element_data,
                                                                       pressure_strong_function)

In [ ]:
def speed_x_strong_function(marker, x, y, z):  
    return speed_x_exact(x, y, z)
def speed_y_strong_function(marker, x, y, z):  
    return speed_y_exact(x, y, z)

u_x_strong = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_mesh_dofs_info,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_x_strong_function)
u_y_strong = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_strong_solution(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_mesh_dofs_info,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_y_strong_function)

In [ ]:
def nu_term(x, y, z):  
    return 1.0
def b_x_term(x, y, z):  
    return np.array([\
        1.0,\
        0.0,\
        0.0])
def b_y_term(x, y, z):  
    return np.array([\
        0.0,\
        1.0,\
        0.0])

A_operator = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_diffusion_operator(geometry_utilities,
                                                                                       mesh,
                                                                                       mesh_geometric_data,
                                                                                       speed_dofs_data,
                                                                                       speed_dofs_data,
                                                                                       speed_reference_element_data,
                                                                                       speed_reference_element_data,
                                                                                       nu_term)


J_A_x = other_ut.make_np_sparse(A_operator.operator_dofs, [tot_dofs, tot_dofs], [0, 0])
J_A_y = other_ut.make_np_sparse(A_operator.operator_dofs, [tot_dofs, tot_dofs], [speed_n_dofs, speed_n_dofs])

B_x_operator = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_advection_operator(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       pressure_dofs_data,
                                                                       speed_reference_element_data,
                                                                       pressure_reference_element_data,
                                                                       b_x_term)
B_y_operator = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_advection_operator(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       pressure_dofs_data,
                                                                       speed_reference_element_data,
                                                                       pressure_reference_element_data,
                                                                       b_y_term)

J_B_x = other_ut.make_np_sparse(B_x_operator.operator_dofs, [tot_dofs, tot_dofs], [2 * speed_n_dofs, 0])
J_B_y = other_ut.make_np_sparse(B_y_operator.operator_dofs, [tot_dofs, tot_dofs], [2 * speed_n_dofs, speed_n_dofs])
J_BT_x = other_ut.make_np_sparse(B_x_operator.operator_dofs, [tot_dofs, tot_dofs], [2 * speed_n_dofs, 0], True)
J_BT_y = other_ut.make_np_sparse(B_y_operator.operator_dofs, [tot_dofs, tot_dofs], [2 * speed_n_dofs, speed_n_dofs], True)

In [ ]:
solution = scipy.sparse.linalg.spsolve(J_A_x + J_A_y - J_B_x - J_B_y - J_BT_x - J_BT_y, J_f)
u_x = solution[0:speed_n_dofs]
u_y = solution[speed_n_dofs:2 * speed_n_dofs]
p = solution[2 * speed_n_dofs:]

In [ ]:
u_x_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       speed_dofs_data,
                                                                                         u_x,
                                                                                         u_x_strong,
                                                                       speed_x_exact)
u_y_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       speed_dofs_data,
                                                                                         u_y,
                                                                                         u_y_strong,
                                                                       speed_y_exact)
p_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       pressure_dofs_data,
                                                                                         p,
                                                                                         p_strong,
                                                                       pressure_exact)

In [ ]:
vtk_utilities.export_solution_2(export_solution_path + '/u_x',
                                mesh, 
                                u_x_on_cell0Ds.numeric_solution)
vtk_utilities.export_solution_2(export_solution_path + '/u_y',
                                mesh, 
                                u_y_on_cell0Ds.numeric_solution)
vtk_utilities.export_solution_2(export_solution_path + '/p',
                                mesh, 
                                p_on_cell0Ds.numeric_solution)
other_ut.plot_solution(mesh, u_x_on_cell0Ds.numeric_solution, "u_x") 
other_ut.plot_solution(mesh, u_y_on_cell0Ds.numeric_solution, "u_y")
other_ut.plot_solution(mesh, np.sqrt(u_x_on_cell0Ds.numeric_solution * u_x_on_cell0Ds.numeric_solution + u_y_on_cell0Ds.numeric_solution * u_y_on_cell0Ds.numeric_solution), "u_mag")
other_ut.plot_solution(mesh, p_on_cell0Ds.numeric_solution, "p") 

Let us define the parameters for the POD, i.e. the snapshot number and the parametric space.

In [ ]:
### define the training set

snapshot_num = 100
mu1_range = [1., 10.]
mu2_range = [1., 3.]
P = np.array([mu1_range, mu2_range])

training_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(snapshot_num, P.shape[0]))




Here, we define the matrices needed to compute the supremizer for each solution of the Stokes problem.

In [ ]:
X_1 = other_ut.make_np_sparse(A_operator.operator_dofs, [2 * speed_n_dofs, 2 * speed_n_dofs], [0, 0])
X_2 = other_ut.make_np_sparse(A_operator.operator_dofs, [2 * speed_n_dofs, 2 * speed_n_dofs], [speed_n_dofs, speed_n_dofs])
B_1 = other_ut.make_np_sparse(B_x_operator.operator_dofs, [pressure_n_dofs, 2 * speed_n_dofs], [0, 0])
B_2 = other_ut.make_np_sparse(B_y_operator.operator_dofs, [pressure_n_dofs, 2 * speed_n_dofs], [0, speed_n_dofs])

We apply the **partitioned POD**. Nemely, we apply a POD for each of the _three_ variables of the Stokes equations.

**Question time!** why three?

If we apply three different PODs, we need three different snapshots matrices.
Usually, computing the supremizer for all the pressure modes retained is not feasable (FE offline), thus, we use the _inexact supremizer_. We compute the supremizers for the snapshots that we already have and we perform the POD on the supremizer snapshots, too. The basis will be added in the projection phase.

Experimentally has been verified that it is a valid strategy to avoid $N_{\delta}$-dependent computations on the online phase.

In [ ]:
#### snapshot matrices creation

snapshot_matrix_u = []
snapshot_matrix_s = [] ### supremizer snapshot
snapshot_matrix_p = []

tol = 1. - 1e-7
N_max = 20

for mu in training_set:
  thetaA1 = mu[0]
  mu_2 = mu[1]
  
  #### the problem is not affine: I have to assemble in this stage!! ###
  f_x = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_reference_element_data,
                                                                       f_x_function)
  f_y = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_reference_element_data,
                                                                       f_y_function)

  J_f = np.concatenate([f_x, f_y, np.zeros(pressure_n_dofs)])
  snapshot_solution = scipy.sparse.linalg.spsolve(J_A_x + J_A_y - J_B_x - J_B_y - J_BT_x - J_BT_y, J_f)    
    
  snapshot_u = snapshot_solution[0:2 * speed_n_dofs]
  snapshot_matrix_u.append(np.copy(snapshot_u))
  
  snapshot_p = snapshot_solution[2 * speed_n_dofs:]
  snapshot_matrix_p.append(np.copy(snapshot_p))

  snapshot_s = scipy.sparse.linalg.spsolve(X_1 + X_2, np.transpose(B_1 + B_2) @ p)
  snapshot_matrix_s.append(np.copy(snapshot_s)) 


snapshot_matrix_u = np.array(snapshot_matrix_u) 

snapshot_matrix_s = np.array(snapshot_matrix_s) 

snapshot_matrix_p = np.array(snapshot_matrix_p) 

  

For the sake of clarity, let us recall the inner products.

**Question time!** why am I defining only the inner product for the velocity?

In [ ]:
inner_product_u = X_1 + X_2

Below, we define a function that, given a covariance matrix (the maximum number of basis functions and a tolerance) computes the related eigenvalues and eigenvectors, returns the eigenvectors and the basis number.

In [ ]:
def eig_analysis(C, N_max=None, tol=1e-9):
  L_e, VM_e = np.linalg.eig(C)
  eigenvalues = []
  eigenvectors = []


  #### check

  for i in range(len(L_e)):
    eig_real = L_e[i].real
    eig_complex = L_e[i].imag
    assert np.isclose(eig_complex, 0.)
    eigenvalues.append(eig_real)
    eigenvectors.append(VM_e[i].real)


  total_energy = sum(eigenvalues)
  retained_energy_vector = np.cumsum(eigenvalues)
  relative_retained_energy = retained_energy_vector/total_energy


  if all(flag==False for flag in relative_retained_energy>= tol) and N_max != None:
    N = N_max
  else:
    N = np.argmax(relative_retained_energy >= tol) + 1
  
  return N, eigenvectors



In [ ]:
### covariance matrix

C_u = snapshot_matrix_u @ inner_product_u @ np.transpose(snapshot_matrix_u)
C_s = snapshot_matrix_s @ inner_product_u @ np.transpose(snapshot_matrix_s)
C_p = snapshot_matrix_p @ np.transpose(snapshot_matrix_p)

N_u, eigs_u = eig_analysis(C_u, N_max=N_max, tol=tol)
N_s, eigs_s = eig_analysis(C_s, N_max=N_max, tol=tol)
N_p, eigs_p = eig_analysis(C_p, N_max=N_max, tol=tol)

print(N_u, N_s, N_p)

Now we create a function that creates the basis, given the snapshots matrix, the reduced dimension and the eigenvectors.

In [ ]:
def create_basis_functions_matrix(N, snapshot_matrix, eigenvectors, inner_product=None):
  
  basis_functions = []
  
  for n in range(N):
    eigenvector =  eigenvectors[n]
    basis = np.transpose(snapshot_matrix)@eigenvector
    if inner_product!= None:
      norm = np.sqrt(np.transpose(basis) @ inner_product @ basis) ## metti inner product
    else:
      norm = np.sqrt(np.transpose(basis) @ basis)
    basis /= norm
    basis_functions.append(np.copy(basis))

  basis_function_matrix = np.transpose(np.array(basis_functions))
  
  return basis_function_matrix

We create three separate basis functions and then the global basis function that we need for the projection:

$$\mathbb {B} = 
\begin{bmatrix} \mathbb B_u \cup \mathbb B_s & 0\\
0 & \mathbb B_p
\end{bmatrix}.$$


In [ ]:
basis_functions_u = create_basis_functions_matrix(N_u, snapshot_matrix_u, eigs_u, inner_product=inner_product_u)
basis_functions_s = create_basis_functions_matrix(N_s, snapshot_matrix_s, eigs_s, inner_product=inner_product_u)
basis_functions_p = create_basis_functions_matrix(N_p, snapshot_matrix_p, eigs_p)

In [ ]:
print(basis_functions_u.shape)
print(basis_functions_p.shape)
print(basis_functions_u.shape[0] + basis_functions_p.shape[0])
print(basis_functions_p.shape)
print(solution.shape)

global_basis_function_matrix = np.zeros((basis_functions_u.shape[0] + basis_functions_p.shape[0],N_u + N_s + N_p))
global_basis_function_matrix[0:basis_functions_u.shape[0], 0:N_u] = basis_functions_u
global_basis_function_matrix[0:basis_functions_u.shape[0], N_u : N_u + N_s] = basis_functions_s
global_basis_function_matrix[basis_functions_u.shape[0]:, N_u + N_s:] = basis_functions_p
print(global_basis_function_matrix.shape)

global_basis_function_matrix_no_sipremizer = np.zeros((basis_functions_u.shape[0] + basis_functions_p.shape[0],N_u + N_p))
global_basis_function_matrix_no_sipremizer[0:basis_functions_u.shape[0], 0:N_u] = basis_functions_u
global_basis_function_matrix_no_sipremizer[basis_functions_u.shape[0]:, N_u:] = basis_functions_p

We now define the assemble-functions

In [ ]:
def assemble_reduced_matrix(basis, fom_matrix):
  return np.transpose(basis) @ (fom_matrix) @ basis

def assemble_reduced_vector(basis, fom_vector):
  return np.transpose(basis) @ (fom_vector)

Let us finish the offline phase

In [ ]:
### ASSEMBLE REDUCED SYSTEMS
reduced_stiff_Stokes = assemble_reduced_matrix(global_basis_function_matrix, (J_A_x + J_A_y)) 
reduced_divergence_operator_1 = assemble_reduced_matrix(global_basis_function_matrix, (J_B_x)) 
reduced_divergence_operator_2 = assemble_reduced_matrix(global_basis_function_matrix, (J_B_y))

We are ready for a new evaluation!

In [ ]:
### New eval
thetaA1 = 1
mu_2 = 2

f_x = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_reference_element_data,
                                                                       f_x_function)
f_y = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                   mesh,
                                                                   mesh_geometric_data,
                                                                   speed_dofs_data,
                                                                   speed_reference_element_data,
                                                                   speed_reference_element_data,
                                                                   f_y_function)

J_f = np.concatenate([f_x, f_y, np.zeros(pressure_n_dofs)])

reduced_lhs = thetaA1*reduced_stiff_Stokes - reduced_divergence_operator_1 - reduced_divergence_operator_2 - np.transpose(reduced_divergence_operator_1) - np.transpose(reduced_divergence_operator_2)
reduced_rhs = assemble_reduced_vector(global_basis_function_matrix, J_f)
print(np.linalg.det(reduced_lhs))

In [ ]:
reduced_solution = np.linalg.solve(reduced_lhs, reduced_rhs)
print(reduced_solution)

In [ ]:
###### plot #######
reduced_u_dof = N_u + N_s
# reduced_p_dof = N_p
reduced_solution_FE_basis = global_basis_function_matrix @ reduced_solution

In [ ]:
reduced_u_x = reduced_solution_FE_basis[0:speed_n_dofs]
reduced_u_y = reduced_solution_FE_basis[speed_n_dofs:2 * speed_n_dofs]
reduced_p = reduced_solution_FE_basis[2 * speed_n_dofs:]

In [ ]:
reduced_u_x_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       speed_dofs_data,
                                                                                         reduced_u_x,
                                                                                         u_x_strong,
                                                                       speed_x_exact)
reduced_u_y_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       speed_dofs_data,
                                                                                         reduced_u_y,
                                                                                         u_y_strong,
                                                                       speed_y_exact)
reduced_p_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       pressure_dofs_data,
                                                                                         reduced_p,
                                                                                         p_strong,
                                                                       pressure_exact)

In [ ]:
vtk_utilities.export_solution_2(export_solution_path + '/u_x',
                                mesh, 
                                reduced_u_x_on_cell0Ds.numeric_solution)
vtk_utilities.export_solution_2(export_solution_path + '/u_y',
                                mesh, 
                                reduced_u_y_on_cell0Ds.numeric_solution)
vtk_utilities.export_solution_2(export_solution_path + '/p',
                                mesh, 
                                reduced_p_on_cell0Ds.numeric_solution)
other_ut.plot_solution(mesh, reduced_u_x_on_cell0Ds.numeric_solution, "u_x") 
other_ut.plot_solution(mesh, reduced_u_y_on_cell0Ds.numeric_solution, "u_y")
other_ut.plot_solution(mesh, np.sqrt(reduced_u_x_on_cell0Ds.numeric_solution * reduced_u_x_on_cell0Ds.numeric_solution + reduced_u_y_on_cell0Ds.numeric_solution * reduced_u_y_on_cell0Ds.numeric_solution), "u_mag")
other_ut.plot_solution(mesh, reduced_p_on_cell0Ds.numeric_solution, "p") 

**What happens without supremizer?**

In [ ]:
### NO SUPREMIZER
reduced_stiff_Stokes_nsup = assemble_reduced_matrix(global_basis_function_matrix_no_sipremizer, (J_A_x + J_A_y)) # np.transpose(global_basis_function_matrix) @ (J_A_x + J_A_y) @ global_basis_function_matrix
reduced_divergence_operator_1_nsup = assemble_reduced_matrix(global_basis_function_matrix_no_sipremizer, (J_BT_x)) #np.transpose(global_basis_function_matrix) @ J_B_x @ global_basis_function_matrix
reduced_divergence_operator_2_nsup = assemble_reduced_matrix(global_basis_function_matrix_no_sipremizer, (J_BT_y))
# - J_BT_x - J_BT_y

In [ ]:
### New eval
reduced_lhs = thetaA1*reduced_stiff_Stokes_nsup - reduced_divergence_operator_1_nsup - reduced_divergence_operator_2_nsup - np.transpose(reduced_divergence_operator_1_nsup) - np.transpose(reduced_divergence_operator_2_nsup)
reduced_rhs = assemble_reduced_vector(global_basis_function_matrix_no_sipremizer, J_f)
print(np.linalg.det(reduced_lhs))

In [ ]:
reduced_solution = np.linalg.solve(reduced_lhs, reduced_rhs)
print(reduced_solution)

In [ ]:
###### plot #######
reduced_u_dof = N_u + N_s
# reduced_p_dof = N_p
reduced_solution_FE_basis = global_basis_function_matrix_no_sipremizer @ reduced_solution

In [ ]:
reduced_u_x = reduced_solution_FE_basis[0:speed_n_dofs]
reduced_u_y = reduced_solution_FE_basis[speed_n_dofs:2 * speed_n_dofs]
reduced_p = reduced_solution_FE_basis[2 * speed_n_dofs:]

In [ ]:
reduced_u_x_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       speed_dofs_data,
                                                                                         reduced_u_x,
                                                                                         u_x_strong,
                                                                       speed_x_exact)
reduced_u_y_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       speed_dofs_data,
                                                                                         reduced_u_y,
                                                                                         u_y_strong,
                                                                       speed_y_exact)
reduced_p_on_cell0Ds = polydim.pde_tools.assembler_utilities.pcc_2_d.extract_solution_on_cell0_ds(mesh,
                                                                       pressure_dofs_data,
                                                                                         reduced_p,
                                                                                         p_strong,
                                                                       pressure_exact)

In [ ]:
vtk_utilities.export_solution_2(export_solution_path + '/u_x',
                                mesh, 
                                reduced_u_x_on_cell0Ds.numeric_solution)
vtk_utilities.export_solution_2(export_solution_path + '/u_y',
                                mesh, 
                                reduced_u_y_on_cell0Ds.numeric_solution)
vtk_utilities.export_solution_2(export_solution_path + '/p',
                                mesh, 
                                reduced_p_on_cell0Ds.numeric_solution)
other_ut.plot_solution(mesh, reduced_u_x_on_cell0Ds.numeric_solution, "u_x") 
other_ut.plot_solution(mesh, reduced_u_y_on_cell0Ds.numeric_solution, "u_y")
other_ut.plot_solution(mesh, np.sqrt(reduced_u_x_on_cell0Ds.numeric_solution * reduced_u_x_on_cell0Ds.numeric_solution + reduced_u_y_on_cell0Ds.numeric_solution * reduced_u_y_on_cell0Ds.numeric_solution), "u_mag")
other_ut.plot_solution(mesh, reduced_p_on_cell0Ds.numeric_solution, "p") 

Let us analyze the usual stuff: errors and speedups!
Below you find a function that computes the error.

In [ ]:
######### def error functions ######

def compute_error(fom_solution, rom_solution_FE_basis, inner_product=None, type_err="relative"):
    
    error_function_u = fom_solution - rom_solution_FE_basis
    
    if inner_product == None:
        inner_product_matrix = np.identity(fom_solution.shape[0])
    else:
      print()
      inner_product_matrix = inner_product
    
    error_norm_squared_component = np.transpose(error_function_u) @ inner_product_matrix @ error_function_u
    absolute_error = np.sqrt(abs(error_norm_squared_component))
    
    if type_err == "absolute":
      
      return absolute_error
    
    else:
      full_solution_norm_squared_component = np.transpose(fom_solution) @  inner_product_matrix @ fom_solution
      relative_error = absolute_error/np.sqrt(abs(full_solution_norm_squared_component))
    
      return relative_error
    

In [ ]:
### compute error
import time

abs_err_u = []
rel_err_u = []

abs_err_p = []
rel_err_p = []

testing_set = np.random.uniform(low=P[:, 0], high=P[:, 1], size=(100, P.shape[0]))
speed_up = []

print("Computing error and speedup analysis")

for mu in testing_set:
  
  thetaA2 = mu[0]
  thetaf1 = mu[1]

  ##### full #####
  thetaA1 = mu[0]
  mu_2 = mu[1]
  
  #### the problem is not affine: I have to assemble in this stage!! ###
  start_time_assemble = time.time()
  f_x = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                       mesh,
                                                                       mesh_geometric_data,
                                                                       speed_dofs_data,
                                                                       speed_reference_element_data,
                                                                       speed_reference_element_data,
                                                                       f_x_function)
  f_y = polydim.pde_tools.assembler_utilities.pcc_2_d.assemble_source_term(geometry_utilities,
                                                                   mesh,
                                                                   mesh_geometric_data,
                                                                   speed_dofs_data,
                                                                   speed_reference_element_data,
                                                                   speed_reference_element_data,
                                                                   f_y_function)

  J_f = np.concatenate([f_x, f_y, np.zeros(pressure_n_dofs)])
  lhs = J_A_x + J_A_y - J_B_x - J_B_y - J_BT_x - J_BT_y
  rhs = J_f
  time_assemble = time.time() - start_time_assemble
  
  start_fom = time.time()
  full_solution = snapshot_solution = scipy.sparse.linalg.spsolve(lhs, rhs)
  time_fom = time.time() - start_fom

  full_solution_u = full_solution[0:2*speed_n_dofs]
  full_solution_p = full_solution[2*speed_n_dofs:]

  #### reduced #####
  print("thetaA1",thetaA1,"thetaA2", thetaA2,"thetaf1", thetaf1,"mu_2", mu_2)

  reduced_lhs = thetaA1*reduced_stiff_Stokes - reduced_divergence_operator_1 - reduced_divergence_operator_2 - np.transpose(reduced_divergence_operator_1) - np.transpose(reduced_divergence_operator_2)
  reduced_rhs = assemble_reduced_vector(global_basis_function_matrix, J_f)

  print(np.linalg.det(reduced_lhs))
  start_rom = time.time()
  reduced_solution = np.linalg.solve(reduced_lhs, reduced_rhs)
  time_rom = time.time() - start_rom
  
  speed_up.append(time_fom/(time_rom+time_assemble))
  
  reduced_solution_FE_basis = global_basis_function_matrix @ reduced_solution
  reduced_u_FE = reduced_solution_FE_basis[0:2*speed_n_dofs]
  reduced_p_FE = reduced_solution_FE_basis[2*speed_n_dofs:]

  ### computing error
  
  abs_err_u_mu = compute_error(full_solution_u, reduced_u_FE, inner_product=inner_product_u, type_err="absolute")
  rel_err_u_mu = compute_error(full_solution_u, reduced_u_FE, inner_product=inner_product_u)
  abs_err_u.append(abs_err_u_mu)
  rel_err_u.append(rel_err_u_mu)

  abs_err_p_mu = compute_error(full_solution_p, reduced_p_FE, type_err="absolute")
  rel_err_p_mu = compute_error(full_solution_p, reduced_p_FE)
  abs_err_p.append(abs_err_p_mu)
  rel_err_p.append(rel_err_p_mu)


In [ ]:
print("avarege relative error for velocity = ", np.mean(rel_err_u) )
print("avarege absolute error for velocity = ", np.mean(abs_err_u) )

print("avarege relative error for pressure = ", np.mean(rel_err_p) )
print("avarege absolute error for pressure = ", np.mean(abs_err_p) )


print("avarege speed_up = ", np.mean(speed_up) )